# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

Se proporciona solamente un generador de laberintos perfectos mediante **Hunt-and-Kill**. A partir del grafo producido, el estudiante deberá diseñar, implementar, verificar y comparar algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El proyecto debe realizarse en tres versiones:

1. **Desde cero:** sin bibliotecas de búsqueda ni de grafos.
2. **SimpleAI:** modelando el laberinto como un `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

> Este cuaderno es deliberadamente incompleto. No contiene implementaciones Python de DFS, BFS, UCS, IDDFS, búsqueda bidireccional ni Lee. El trabajo evaluado consiste en convertir las especificaciones y el pseudocódigo en soluciones propias.

## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [ ]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### Respuestas teóricas — Auditoría del generador Hunt-and-Kill

#### 1. Fases Hunt y Kill en el código suministrado

El algoritmo construye un árbol generador (spanning tree) sobre la cuadrícula transitando entre dos fases:

- **Fase Inicial (Líneas 139-142):** Se define `no_visitadas` con la totalidad de celdas $|V| = \text{filas} \times \text{columnas}$. Se escoge una celda aleatoria `actual` que se remueve de `no_visitadas`, actuando como raíz del laberinto.
- **Fase KILL (Caminata Aleatoria / Random Walk, Líneas 144-156):**
  ```python
  libres = [vecino for vecino in self.vecinos_geometricos(actual) if vecino in no_visitadas]
  if libres:
      siguiente = self.rng.choice(libres)
      self.conectar(actual, siguiente)
      no_visitadas.remove(siguiente)
      actual = siguiente
      continue
  ```
  Desde la celda `actual`, el algoritmo busca celdas adyacentes ortogonales que aún no formen parte del árbol (`vecino in no_visitadas`). Si existen, elige una celda al azar (`siguiente`), establece un pasillo bidireccional (`conectar`), la añade al árbol retirándola de `no_visitadas` y avanza a ella. Esta fase simula "cavar un túnel" de forma continua hasta quedar acorralado.
- **Fase HUNT (Búsqueda y Conexión, Líneas 158-173):**
  ```python
  candidatas = []
  for celda in no_visitadas:
      visitados = [vecino for vecino in self.vecinos_geometricos(celda) if vecino not in no_visitadas]
      if visitados:
          candidatas.append((celda, visitados))
  actual, visitados = self.rng.choice(candidatas)
  vecino = self.rng.choice(visitados)
  self.conectar(actual, vecino)
  no_visitadas.remove(actual)
  ```
  Al alcanzar un callejón sin salida (`libres` vacío), el algoritmo escanea las celdas no visitadas (`celda in no_visitadas`) buscando aquellas que tengan al menos un vecino ortogonal que ya pertenezca al árbol (`vecino not in no_visitadas`). Selecciona una celda candidata, la conecta a dicho árbol mediante una arista, la elimina de `no_visitadas` y la fija como nueva `actual` para reanudar de inmediato la fase Kill en la siguiente iteración.

#### 2. Demostración formal de que el grafo generado es conexo

Sea $T_k = (V_k, E_k)$ el subgrafo construido hasta el paso $k$, donde $V_k$ es el conjunto de celdas extraídas de `no_visitadas`.
- **Base inductiva ($k=0$):** Al iniciar, $V_0 = \{actual_0\}$ y $E_0 = \emptyset$. Un grafo trivial de un solo vértice es conexo por definición.
- **Paso inductivo ($k \to k+1$):** Asumimos que $T_k$ es conexo. En el paso $k+1$, una celda $u \in \text{no\_visitadas}$ se incorpora al árbol:
  - En la fase **Kill**, $u$ se conecta mediante la arista $(actual, u)$, donde $actual \in V_k$. Dado que por hipótesis inductiva existe un camino simple entre cualquier nodo $v \in V_k$ y $actual$, el camino $v \leadsto actual - u$ conecta a $v$ con $u$. Por ende, $T_{k+1}$ es conexo.
  - En la fase **Hunt**, $u$ se conecta con un $vecino \in V_k$. Análogamente, para todo $v \in V_k$, la existencia del camino $v \leadsto vecino$ garantiza el camino $v \leadsto vecino - u$, manteniendo $T_{k+1}$ conexo.
- **Conclusión:** Dado que el bucle `while no_visitadas:` se ejecuta hasta que $\text{no\_visitadas} = \emptyset$, el conjunto final es $V = V_{|V|-1}$. Por inducción matemática, el grafo final $G = (V, E)$ es estrictamente conexo.

#### 3. Justificación de que contiene exactamente $|V|-1$ aristas

Inicialmente $|V_0| = 1$ y $|E_0| = 0$.
En cada paso de extensión (sea en Kill o en Hunt):
1. Se extrae exactamente una celda de `no_visitadas` (`no_visitadas.remove(siguiente)` o `no_visitadas.remove(actual)`), incrementando el orden del subgrafo en 1: $|V_{k+1}| = |V_k| + 1$.
2. Se invoca exactamente una vez `self.conectar(a, b)`, creando una única arista no dirigida: $|E_{k+1}| = |E_k| + 1$.
Para agotar las $|V|$ celdas partiendo de 1 celda inicial, se requieren exactamente $|V| - 1$ adiciones. Por lo tanto, el número total de aristas no dirigidas es invariablemente $|E| = |V| - 1$.

#### 4. Implicación de que es un Árbol

De acuerdo con el teorema fundamental de caracterización de árboles en teoría de grafos, para cualquier grafo simple no dirigido $G = (V, E)$ con $|V| < \infty$, las siguientes propiedades son equivalentes:
1. $G$ es un árbol (grafo conexo y acíclico).
2. $G$ es conexo y tiene $|E| = |V| - 1$.
3. $G$ es acíclico y tiene $|E| = |V| - 1$.
4. Existe un único camino simple entre cualquier par de vértices en $V$.

Habiendo demostrado formalmente en (2) que $G$ es conexo y en (3) que posee exactamente $|E| = |V| - 1$ aristas, queda rigurosamente demostrado que $G$ es un árbol (laberinto perfecto sin islas aisladas ni ciclos/bucles).


In [ ]:
# Verificación y Auditoría Automática del Generador Hunt-and-Kill
from collections import deque
import random

def auditar_laberinto(laberinto_cls, filas=20, columnas=20, semilla=2026):
    lab = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla)
    grafo = lab.generar()
    total_nodos = filas * columnas
    
    print(f"=== AUDITORÍA AUTOMÁTICA DEL GENERADOR ({filas}x{columnas}, semilla={semilla}) ===\n")
    
    # 1. Simetría de adyacencias
    simetrico = True
    for u, vecinos in grafo.items():
        for v in vecinos:
            if u not in grafo[v]:
                simetrico = False
                break
    print(f"[1] Simetría de adyacencias: {'✓ CORRECTO' if simetrico else '✗ FALLÓ'}")
    assert simetrico, "Error: Hay aristas no dirigidas asimétricas."
    
    # 2. Ausencia de diagonales (ortogonalidad)
    ortogonal = True
    for u, vecinos in grafo.items():
        for v in vecinos:
            manhattan = abs(u[0] - v[0]) + abs(u[1] - v[1])
            if manhattan != 1:
                ortogonal = False
                break
    print(f"[2] Ausencia de diagonales (distancia Manhattan = 1): {'✓ CORRECTO' if ortogonal else '✗ FALLÓ'}")
    assert ortogonal, "Error: Se detectaron conexiones diagonales o no unitarias."
    
    # 3. Validez de coordenadas
    coordenadas_validas = all(
        0 <= r < filas and 0 <= c < columnas
        for r, c in grafo
        for vr, vc in grafo[(r, c)]
        if 0 <= vr < filas and 0 <= vc < columnas
    )
    print(f"[3] Validez de coordenadas dentro de [0, {filas})x[0, {columnas}): {'✓ CORRECTO' if coordenadas_validas else '✗ FALLÓ'}")
    assert coordenadas_validas, "Error: Coordenadas fuera de rango."
    
    # 4. Conectividad total (BFS)
    inicio = next(iter(grafo))
    visitados = set([inicio])
    cola = deque([inicio])
    while cola:
        curr = cola.popleft()
        for vec in grafo[curr]:
            if vec not in visitados:
                visitados.add(vec)
                cola.append(vec)
    conexo = (len(visitados) == total_nodos)
    print(f"[4] Conectividad global (|V_alcanzados| = {len(visitados)}/{total_nodos}): {'✓ CORRECTO' if conexo else '✗ FALLÓ'}")
    assert conexo, f"Error: Grafo desconexo ({len(visitados)}/{total_nodos} nodos alcanzados)."
    
    # 5. Ausencia de ciclos y fórmula de árbol (|E| == |V| - 1)
    num_aristas = sum(len(vecinos) for vecinos in grafo.values()) // 2
    es_arbol_aristas = (num_aristas == total_nodos - 1)
    
    # Detección explícita de ciclos vía DFS
    hay_ciclo = False
    padres = {}
    visitados_dfs = set()
    pila = [(inicio, None)]
    while pila:
        curr, p = pila.pop()
        if curr in visitados_dfs:
            continue
        visitados_dfs.add(curr)
        padres[curr] = p
        for vec in grafo[curr]:
            if vec == p:
                continue
            if vec in visitados_dfs:
                hay_ciclo = True
                break
            pila.append((vec, curr))
        if hay_ciclo:
            break
            
    print(f"[5] Ausencia de ciclos (|E| = {num_aristas}, |V|-1 = {total_nodos-1}, ciclos={hay_ciclo}): {'✓ CORRECTO' if (es_arbol_aristas and not hay_ciclo) else '✗ FALLÓ'}")
    assert es_arbol_aristas and not hay_ciclo, "Error: El grafo contiene ciclos o viola |E| = |V| - 1."
    
    # 6. Reproducibilidad con semilla
    lab_rep = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla)
    grafo_rep = lab_rep.generar()
    reproducible = (grafo == grafo_rep)
    
    lab_dif = laberinto_cls(filas=filas, columnas=columnas, semilla=semilla + 99)
    grafo_dif = lab_dif.generar()
    sensible_semilla = (grafo != grafo_dif)
    
    print(f"[6] Reproducibilidad por semilla: {'✓ CORRECTO' if (reproducible and sensible_semilla) else '✗ FALLÓ'}")
    assert reproducible and sensible_semilla, "Error en reproducibilidad de semillas."
    
    print("\n>>> TODAS LAS PROPIEDADES FUERON VERIFICADAS Y AUDITADAS EXITOSAMENTE <<<")

# Ejecutar auditoría sobre el generador suministrado
auditar_laberinto(LaberintoHuntKill, filas=20, columnas=20, semilla=2026)


## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

### Parámetros asignados

| Parámetro | Valor |
|---|---|
| Código estudiantil | `202410057` |
| Semilla | `10057` |
| Filas ($20 + 7$) | `27` |
| Columnas ($20 + 5$) | `25` |
| Inicio (Cuadrante Superior Izquierdo) | `(3, 4)` |
| Meta (Cuadrante Inferior Derecho) | `(23, 20)` |

> *Nota:* El código está parametrizado mediante variables explícitas, lo que permite al docente modificar cualquiera de estos valores en vivo durante la sustentación sin alterar la lógica de búsqueda.

In [ ]:
# Construcción reproducible de la instancia individual
CODIGO_ESTUDIANTIL = 202410057
SEMILLA = int(str(CODIGO_ESTUDIANTIL)[-5:])       # 10057
FILAS = 20 + int(str(CODIGO_ESTUDIANTIL)[-1])       # 27
COLUMNAS = 20 + int(str(CODIGO_ESTUDIANTIL)[-2])    # 25

# Cuadrantes opuestos: Superior Izquierdo e Inferior Derecho
INICIO = (3, 4)
META = (FILAS - 4, COLUMNAS - 5)  # (23, 20)

# Instanciar y generar el laberinto
laberinto_individual = LaberintoHuntKill(filas=FILAS, columnas=COLUMNAS, semilla=SEMILLA)
grafo_individual = laberinto_individual.generar()

# Verificación de cuadrantes
def obtener_cuadrante(celda, filas, columnas):
    r, c = celda
    vertical = "Norte (Superior)" if r < filas / 2 else "Sur (Inferior)"
    horizontal = "Oeste (Izquierdo)" if c < columnas / 2 else "Este (Derecho)"
    return f"{vertical} - {horizontal}"

print("=== INSTANCIA INDIVIDUAL CONFIGURADA ===")
print(f"Código: {CODIGO_ESTUDIANTIL}")
print(f"Semilla: {SEMILLA}")
print(f"Dimensiones: {FILAS} filas x {COLUMNAS} columnas ({FILAS * COLUMNAS} celdas)")
print(f"Inicio: {INICIO} -> Cuadrante: {obtener_cuadrante(INICIO, FILAS, COLUMNAS)}")
print(f"Meta:   {META} -> Cuadrante: {obtener_cuadrante(META, FILAS, COLUMNAS)}")
print(f"Total aristas en el árbol: {sum(len(v) for v in grafo_individual.values()) // 2} (|V|-1)")


## 3. Formulación formal del problema

Siguiendo el estándar de Russell & Norvig (AIMA, Cap. 3), el entorno se formaliza rigurosamente como un espacio de estados:

| Componente | Especificación formal |
|---|---|
| **Conjunto de estados ($S$)** | $S = \{(r, c) \in \mathbb{Z}^2 \mid 0 \le r < \text{filas} \land 0 \le c < \text{columnas}\}$. Estados inmutables representados como tuplas `(int, int)`. |
| **Estado inicial ($s_0$)** | $s_0 = (r_{\text{inicio}}, c_{\text{inicio}}) = (3, 4) \in S$. |
| **Acciones aplicables ($\text{Actions}(s)$)** | $\text{Actions}(s) = \{a \in \{\text{'ARRIBA'}, \text{'ABAJO'}, \text{'IZQUIERDA'}, \text{'DERECHA'}\} \mid s + \Delta(a) \in \text{grafo}[s]\}$. |
| **Modelo de transición ($\text{Result}(s, a)$)** | $\text{Result}(s, a) = s + \Delta(a)$, determinista y libre de efectos colaterales. |
| **Prueba de objetivo ($\text{Goal-test}(s)$)** | $\text{Goal-test}(s) \iff s == s_{\text{meta}} = (23, 20)$. |
| **Costo de paso ($c(s, a, s')$)** | $c(s, a, s') = 1.0$ para el laberinto estándar con costo unitario; $w(s, s') > 0$ en la variante ponderada. |
| **Costo de una solución ($C$)** | $C = \sum_{i=1}^{k} c(s_{i-1}, a_i, s_i)$. En costos unitarios equivale a la longitud del camino en aristas ($k$). |

Donde los vectores de desplazamiento $\Delta(a)$ corresponden a:
$$\Delta(\text{'ARRIBA'}) = (-1, 0), \quad \Delta(\text{'ABAJO'}) = (1, 0), \quad \Delta(\text{'IZQUIERDA'}) = (0, -1), \quad \Delta(\text{'DERECHA'}) = (0, 1)$$

---

### Preguntas conceptuales obligatorias

#### 1. Diferencia entre estado, nodo de búsqueda y celda dibujada
- **Estado:** Es la configuración abstracta del mundo en un instante dado (en este dominio, la tupla inmutable de coordenadas `(fila, columna)`). No posee noción de pasado, camino ni costo acumulado.
- **Nodo de búsqueda:** Es una estructura de datos generada por el algoritmo durante el proceso de búsqueda para construir el árbol de exploración. Contiene:
  1. `estado`: el estado que representa;
  2. `padre`: apuntador al nodo antecesor;
  3. `accion`: la acción ejecutada en el padre para llegar a este estado;
  4. `costo_camino` ($g(n)$): la suma acumulada de los costos de paso desde la raíz;
  5. `profundidad` ($d(n)$): el número de pasos desde el nodo inicial.
- **Celda dibujada:** Es la entidad gráfica (píxel, parche rectangular o glifo de texto) renderizada en la interfaz visual o gráfica de Matplotlib. No forma parte del modelo algorítmico.

#### 2. ¿Cuándo dos nodos distintos pueden contener el mismo estado?
Dos nodos de búsqueda $n_1 \neq n_2$ pueden contener exactamente el mismo estado $s$ cuando el algoritmo descubre dos caminos alternativos para alcanzar dicha celda. Esto sucede comúnmente en grafos con ciclos o en búsqueda en árbol (donde el mismo estado se re-genera a través de trayectorias distintas con diferente padre, profundidad o costo acumulado $g(n)$).

#### 3. ¿Por qué debe usarse búsqueda en grafo en vez de búsqueda en árbol?
En una cuadrícula o laberinto, cada conexión entre celdas es bidireccional: si existe corredor entre $u$ y $v$, la acción opuesta regresa a $u$. Una **búsqueda en árbol** (que no registra estados visitados) incurre inmediatamente en oscilaciones infinitas ($u \leftrightarrow v$) o en una explosión combinatoria exponencial redundante ($O(b^d)$) reexplorando las mismas celdas una y otra vez. La **búsqueda en grafo** mantiene una colección de estados alcanzados/explorados, garantizando que cada celda sea expandida a lo sumo una vez y acotando el espacio de estados a $|V|$ celdas.

#### 4. Invariantes que deben cumplirse durante una búsqueda
1. **Invariante de Separación de la Frontera:** En todo momento, la frontera separa el conjunto de nodos expandidos/explorados de los estados aún no descubiertos. Todo camino desde el estado inicial a un estado no descubierto debe cruzar la frontera.
2. **Invariante de Trazabilidad del Camino:** Para todo nodo $n$, siguiendo la cadena de punteros `n.padre` hacia atrás se reconstruye un camino válido y conexo de estados continuos hasta $s_0$.
3. **Invariante de Consistencia de Costo:** Para todo nodo $n$, su valor $g(n)$ coincide exactamente con la suma de costos de paso de la trayectoria definida por sus antecesores.
4. **Invariante de Monotonía en UCS:** Cuando los costos de paso son no negativos ($c \ge \epsilon > 0$), la secuencia de costos $g(n)$ de los nodos extraídos de la cola de prioridad es monótonamente no decreciente.


## 4. Contrato común de las tres versiones

Todas las versiones y algoritmos (Desde cero, SimpleAI y AIMA-Python) deben satisfacer estrictamente el siguiente contrato de registro:

```python
@dataclass
class ResultadoBusqueda:
    encontrado: bool            # True si se alcanzó la meta, False en caso contrario
    camino: list[tuple[int,int]]# Secuencia completa de estados [inicio, ..., meta]
    acciones: list[str]         # Secuencia de acciones ['ABAJO', 'DERECHA', ...]
    costo: float                # Costo total acumulado del camino encontrado
    profundidad: int            # Profundidad del nodo meta en el árbol de búsqueda
    expandidos: int             # Nodos retirados de la frontera cuyos sucesores fueron generados
    generados: int              # Total de nodos sucesores construidos
    repetidos_descartados: int  # Sucesores descartados por ya pertenecer a visitados/alcanzados
    frontera_maxima: int        # Tamaño máximo simultáneo alcanzado por la frontera
    tiempo_ms: float            # Tiempo de ejecución en milisegundos
```

### Convenciones obligatorias aplicadas en el taller

- **Expandidos:** Se incrementa únicamente al retirar un nodo de la frontera y proceder a generar sus sucesores. Si el nodo meta es extraído y la búsqueda finaliza de inmediato, dicho nodo **no** se cuenta como expandido.
- **Generados:** Todo nodo sucesor construido se cuenta como generado, independientemente de que luego sea aceptado o descartado por duplicación. El nodo inicial se cuenta como el primer nodo generado ($1$).
- **Repetidos descartados:** Todo sucesor cuyo estado ya se encuentre en la estructura de alcanzados/cerrados (o en frontera con mejor costo) y no reemplace una entrada previa.
- **Frontera máxima:** Mide el pico de ocupación en memoria de la colección de frontera durante toda la ejecución.
- **Camino:** Lista ordenada conteniendo inicio y meta consecutivamente. Si no hay solución, `camino = []` y `encontrado = False`.


## 5. Pseudocódigo base y diseño de búsqueda en grafo

El algoritmo general de búsqueda en grafo desacopla la política de la frontera de la lógica del espacio de estados:

```text
BUSQUEDA-GRAFO(problema, frontera):
    nodo_inicial ← crear_nodo(problema.estado_inicial)
    insertar(frontera, nodo_inicial)
    alcanzados ← {problema.estado_inicial: nodo_inicial.costo}

    mientras frontera no esté vacía:
        nodo ← extraer_segun_politica(frontera)
        si ES_META(nodo.estado):
            devolver RECONSTRUIR(nodo)

        expandidos ← expandidos + 1
        para cada accion en ACCIONES_APLICABLES(nodo.estado):
            hijo ← CONSTRUIR_SUCESOR(nodo, accion)
            generados ← generados + 1
            si hijo.estado no en alcanzados o hijo.costo < alcanzados[hijo.estado]:
                alcanzados[hijo.estado] ← hijo.costo
                insertar(frontera, hijo)
            si no:
                repetidos_descartados ← repetidos_descartados + 1

    devolver FRACASO
```

### Respuestas a las preguntas de diseño

1. **¿Cuándo debe marcarse un estado: al generarlo o al expandirlo?**
   - En **BFS en grafos sin costos (o costo unitario)**, la prueba de meta y el marcado en `alcanzados` pueden realizarse **al generar** el sucesor (verificación anticipada / *early goal test*). Esto ahorra almacenar niveles enteros en la frontera y reduce drásticamente el espacio en memoria sin perder optimalidad.
   - En **UCS (Costo Uniforme)** y problemas con costos variables, la meta debe verificarse **al expandir** (al extraer de la cola de prioridad / *late goal test*), ya que un nodo meta podría generarse inicialmente por un camino subóptimo de alto costo y luego descubrirse un camino más económico.
   - En **DFS**, marcar al expandir o al generar influye en la exploración, pero en grafos no dirigidos marcar al generar evita meter ciclos redundantes a la pila.

2. **¿Qué información mínima debe almacenar cada nodo?**
   La tupla de 5 elementos: `(estado, padre, accion, costo_camino, profundidad)`.

3. **¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?**
   Cada nodo guarda únicamente un apuntador o referencia a su objeto `padre`. Al alcanzar la meta, se itera siguiendo los apuntadores `actual = actual.padre` acumulando estados y acciones en una lista hasta llegar a la raíz (`None`), y finalmente se invierte la lista ($O(d)$ en tiempo y memoria, en vez de $O(d^2)$ que implicaría copiar la lista en cada ramificación).

4. **¿Cómo se resuelve un camino más barato hacia un estado ya descubierto?**
   Se mantiene un diccionario `alcanzados: dict[Estado, float]` que asocia cada estado a su mejor costo $g(s)$ conocido. Si un nuevo nodo genera el mismo estado con un costo $g_{nuevo} < alcanzados[s]$, se actualiza `alcanzados[s] = g_{nuevo}` y se inserta el nuevo nodo en la frontera. En colas de prioridad basadas en heap (donde reordenar elementos internos es costoso), se utiliza la estrategia de **entradas obsoletas (lazy deletion)**: cuando un nodo es extraído del heap, si su costo es estrictamente mayor que el mejor costo registrado en `alcanzados`, se descarta silenciosamente.


# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# VERSIÓN 1: implemente aquí su solución desde cero.

## 7. Lee: propagación de un frente luminoso

No se modela un único rayo que rebota. Se modela una onda que avanza simultáneamente por todos los corredores accesibles.

```text
LEE(grafo, inicio, meta):
    etiqueta[inicio] ← 0
    frente_actual ← {inicio}

    mientras frente_actual no esté vacío y meta no tenga etiqueta:
        frente_siguiente ← vacío
        para cada celda del frente_actual:
            para cada vecino transitable sin etiqueta:
                etiqueta[vecino] ← etiqueta[celda] + 1
                registrar procedencia o dirección
                agregar vecino al frente_siguiente
        frente_actual ← frente_siguiente

    si meta no tiene etiqueta: devolver fracaso
    reconstruir desde meta siguiendo etiquetas decrecientes
```

### Requisitos adicionales

- Dibuje al menos ocho instantes del frente de onda.
- Use una escala de color que represente el tiempo de llegada.
- Compruebe que la etiqueta de la meta coincide con la profundidad de BFS.
- Explique formalmente por qué Lee es BFS en un grafo no ponderado.
- Analice qué deja de funcionar cuando los costos no son unitarios.

In [ ]:
# Implemente y visualice aquí el algoritmo de Lee.

# Versión 2 — SimpleAI

Instalación orientativa:

```text
pip install simpleai
```

Consulte la documentación oficial. Debe crear una subclase de `SearchProblem` y definir, como mínimo:

| Método de SimpleAI | Responsabilidad en el laberinto |
|---|---|
| `actions(state)` | Producir únicamente movimientos legales |
| `result(state, action)` | Calcular el estado sucesor sin mutar el original |
| `is_goal(state)` | Reconocer la celda meta |
| `cost(state, action, state2)` | Devolver el costo del corredor |

Instancie el problema con un estado inicial inmutable y use `graph_search=True`.

Debe ejecutar y estudiar:

- `depth_first`;
- `breadth_first`;
- `uniform_cost`;
- `limited_depth_first`;
- `iterative_limited_depth_first`.

### Exigencia avanzada

SimpleAI devuelve un nodo solución, pero las métricas del contrato no aparecen automáticamente con la misma definición. Investigue su mecanismo de *viewer* o diseñe instrumentación externa **sin modificar el código fuente instalado de la biblioteca**. Documente cualquier diferencia entre las métricas de SimpleAI y las de su versión.

No copie la estructura de su buscador desde cero dentro de esta versión: el propósito es construir correctamente el adaptador del problema y comprender la biblioteca.

In [ ]:
# VERSIÓN 2: modele el problema y ejecute SimpleAI.

# Versión 3 — AIMA-Python

Repositorio de referencia: `aimacode/aima-python`.

Debe modelar el laberinto como una subclase de `Problem`. Investigue y documente los contratos de:

| Método de AIMA-Python | Decisión requerida |
|---|---|
| `actions(state)` | Formato de las acciones y orden determinista |
| `result(state, action)` | Transición entre celdas |
| `goal_test(state)` | Prueba de objetivo |
| `path_cost(c, state1, action, state2)` | Acumulación de costos |

Compare al menos:

- `depth_first_graph_search`;
- `breadth_first_graph_search`;
- `uniform_cost_search`;
- `depth_limited_search`;
- `iterative_deepening_search`;
- `bidirectional_search`.

### Exigencia avanzada

Instrumente los algoritmos sin alterar permanentemente el repositorio. Puede utilizar una subclase del problema, envoltorios o contadores cuidadosamente definidos. Explique por qué contar llamadas a `actions` no siempre coincide con contar nodos generados.

Compare la representación del nodo de AIMA-Python con su propia clase de nodo.

In [ ]:
# VERSIÓN 3: adapte el problema a AIMA-Python.

## 8. Pruebas de aceptación

El estudiante debe implementar pruebas automáticas equivalentes a las siguientes especificaciones, sin recibir su solución:

### Validez de un camino

Para cada resultado exitoso:

1. el primer estado es el inicio;
2. el último estado es la meta;
3. cada pareja consecutiva aparece como arista en el grafo;
4. el costo reportado coincide con la suma de costos;
5. aplicar las acciones reproduce exactamente los estados.

### Concordancia entre versiones

- En costos unitarios: `longitud(BFS) = longitud(UCS) = etiqueta_meta(Lee)`.
- En un árbol: todas las soluciones válidas contienen la misma secuencia de estados.
- Con ciclos: `longitud(BFS) ≤ longitud(DFS)` para una DFS que encuentre solución.
- Con costos positivos: `costo(UCS) ≤ costo(cualquier solución BFS)`.
- Las tres versiones deben concordar en existencia de solución y costo óptimo.

### Casos límite obligatorios

- inicio igual a meta;
- laberinto de `1×1`;
- meta inalcanzable después de eliminar corredores;
- múltiples entradas obsoletas en UCS;
- dos posibles puntos de encuentro bidireccional;
- límite exacto, insuficiente y excesivo en búsqueda limitada.

In [ ]:
# Escriba aquí su batería de pruebas.

## 9. Segunda familia de problemas: ciclos y costos

Hunt-and-Kill produce un árbol. Allí existe un único camino entre dos estados y no se aprecia plenamente la optimalidad. Diseñe, sin modificar el generador original, dos transformaciones:

### A. Laberinto con ciclos

Abra aleatoriamente entre 5% y 12% de las paredes internas que no sean corredores. Debe conservar:

- adyacencias ortogonales;
- simetría del grafo;
- reproducibilidad;
- al menos un ciclo comprobable.

### B. Laberinto ponderado

Asigne costos positivos a corredores o terrenos. Construya deliberadamente un caso donde:

```text
camino con menos pasos ≠ camino de menor costo
```

No use pesos negativos. Justifique si los costos pertenecen a celdas, acciones o aristas y mantenga esa decisión en las tres versiones.

In [ ]:
# Implemente aquí ciclos y costos; no modifique la clase suministrada.

## 10. Diseño experimental

Ejecute como mínimo **30 instancias** por configuración y reporte mediana y rango intercuartílico, no solamente un caso.

| Factor | Niveles mínimos |
|---|---|
| Tamaño | 15×15, 25×25, 40×40 |
| Topología | árbol, 5% ciclos, 10% ciclos |
| Costos | unitarios, ponderados |
| Posición | esquinas, interior, cercanas |
| Algoritmos | todos los exigidos que sean aplicables |

Debe controlar semillas y separar tiempo de generación del tiempo de búsqueda.

### Gráficas obligatorias

1. expandidos frente a número de estados;
2. frontera máxima frente a profundidad de solución;
3. tiempo frente a tamaño;
4. costo y longitud de las soluciones;
5. mapa de calor de Lee;
6. comparación cruzada de las tres versiones.

### Discusión obligatoria

- ¿Qué métrica resulta más estable que el tiempo?
- ¿Cuándo la búsqueda bidireccional pierde su ventaja?
- ¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?
- ¿Qué efecto tiene el orden de sucesores sobre DFS?
- ¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?

In [ ]:
# Construya aquí su protocolo experimental, tablas y gráficas.

## 11. Análisis teórico

Complete y justifique esta tabla usando $b$ como factor de ramificación, $d$ como profundidad de la solución menos profunda y $m$ como profundidad máxima.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | | | | | |
| BFS | | | | | |
| UCS | | | | | |
| DLS | | | | | |
| IDDFS | | | | | |
| Bidireccional | | | | | |
| Lee | | | | | |

Relacione después las cotas teóricas con las mediciones. Una tabla memorizada sin conexión con los experimentos no recibe puntaje completo.

## 12. Preguntas para la defensa

El docente seleccionará algunas al azar:

1. Muestre en memoria la diferencia entre frontera y alcanzados.
2. Cambie BFS a DFS modificando solamente la política apropiada.
3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo.
4. Explique por qué terminar UCS al generar la meta puede ser incorrecto.
5. Reconstruya un camino usando padres sin almacenar caminos completos.
6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres.
7. Explique por qué Lee no es un rayo geométrico.
8. Adapte el problema a una meta múltiple.
9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse.
10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python.
11. Diagnostique una discrepancia de métricas entre dos versiones.
12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta.

## 13. Entrega

El cuaderno final debe contener:

1. auditoría del generador;
2. formulación formal;
3. versión desde cero;
4. versión SimpleAI;
5. versión AIMA-Python;
6. pruebas automáticas;
7. extensión con ciclos y costos;
8. protocolo experimental y gráficas;
9. análisis teórico;
10. conclusiones y referencias;
11. enlace al historial de trabajo o bitácora incorporada.

Todo el cuaderno debe ejecutarse desde el inicio en un entorno limpio. Las celdas fuera de orden, dependencias implícitas y resultados pegados manualmente se consideran defectos reproducibles.

### Rúbrica

| Criterio | Peso |
|---|---:|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% |
| Versión desde cero y estructuras de datos | 25% |
| Adaptación correcta a SimpleAI | 12% |
| Adaptación correcta a AIMA-Python | 13% |
| Pruebas, casos límite y concordancia | 15% |
| Experimentos, métricas y visualización | 12% |
| Análisis teórico y conclusiones | 8% |
| Defensa, trazabilidad y calidad del cuaderno | 5% |

Una solución que produzca un camino pero no satisfaga el contrato, las pruebas o la defensa no se considera completa.

## 14. Referencias de partida

- Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª edición, capítulo 3.
- SimpleAI, documentación de problemas y búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
- SimpleAI, implementación oficial de búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/simpleai/search/traditional.py>
- AIMA-Python, repositorio oficial: <https://github.com/aimacode/aima-python>
- AIMA-Python, módulo de búsqueda: <https://github.com/aimacode/aima-python/blob/master/aima/search.py>
- C. Y. Lee, “An Algorithm for Path Connections and Its Applications”, 1961.
- A. Adamatzky, “Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm”, 2016: <https://arxiv.org/abs/1601.04672>

Las referencias orientan el estudio de las interfaces; no sustituyen la explicación de las decisiones tomadas.